# Lesson 1.4 — CLI Tools for Ops

**Level: Beginner**

## Learning Objectives
- Build multi-command CLIs with **Click** (the industry standard for Python CLIs)
- Use **Rich** to produce structured, colorized terminal output
- Add sub-commands, options, arguments, and choices
- Test CLI commands with `click.testing.CliRunner`
- Write a CLI to disk as a standalone script

**Produces:** `ops_cli.py` — a multi-command ops CLI.

*Official refs:*
- *[Click documentation](https://click.palletsprojects.com/en/8.x/)*
- *[Rich documentation](https://rich.readthedocs.io/)*

## Concepts

### Why Click?

Python's built-in `argparse` is powerful but verbose. **Click** decorators turn
a plain function into a CLI command with zero boilerplate:

```python
@click.command()
@click.option("--name", default="World")
def hello(name):
    click.echo(f"Hello, {name}!")
```

### Click Anatomy

```
@click.group()     → creates a command group  (like: git, docker, kubectl)
@cli.command()     → adds a sub-command       (like: git commit, docker ps)
@click.option()    → named flag               --env staging
@click.argument()  → positional arg           deploy web-api
@click.pass_context→ share state between commands
```

From the [Click quickstart](https://click.palletsprojects.com/en/8.x/quickstart/):

> *"Click tries to make the experience of writing command line tools fun and profitable
> while also preventing any frustration caused by the inability to implement an
> intended CLI API."*

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "click", "rich", "--quiet"],
               capture_output=True)

import click
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from click.testing import CliRunner

console = Console()
print(f"click version: {click.__version__}")

## Code-Along: Step 1 — Rich Terminal Output

In [ ]:
# Rich produces beautiful terminal output with zero effort
# Reference: https://rich.readthedocs.io/en/stable/

# Colored status messages
console.print("[bold green]✓ Deployment succeeded[/bold green]")
console.print("[bold yellow]⚠ Warning: CPU at 82%[/bold yellow]")
console.print("[bold red]✗ Error: Connection refused[/bold red]")

# Rich Table — perfect for resource inventories
table = Table(title="EC2 Inventory", header_style="bold cyan")
table.add_column("Instance ID", style="dim")
table.add_column("Name")
table.add_column("State")
table.add_column("Type")
table.add_column("Region")

table.add_row("i-0abc1234", "web-server-1", "[green]running[/green]", "t3.micro",  "us-east-1")
table.add_row("i-0def5678", "worker-1",     "[green]running[/green]", "t3.small",  "us-east-1")
table.add_row("i-0ghi9012", "db-backup",    "[yellow]stopped[/yellow]","r6i.large","us-east-1")

console.print(table)

# Rich Panel — for highlighted summaries
console.print(Panel(
    "[bold]3[/bold] instances | [yellow]1 stopped[/yellow] needs attention.",
    title="[bold cyan]Health Summary[/bold cyan]", border_style="cyan"
))

## Code-Along: Step 2 — Click CLI Group with Sub-Commands

In [ ]:
@click.group()
@click.option("--verbose", "-v", is_flag=True, help="Enable verbose output")
@click.pass_context
def ops_cli(ctx, verbose):
    """Ops Automation Platform CLI."""
    ctx.ensure_object(dict)
    ctx.obj["verbose"] = verbose

@ops_cli.command()
@click.option("--env", "-e", default="development",
              type=click.Choice(["development", "staging", "production"]),
              help="Target environment")
@click.pass_context
def status(ctx, env):
    """Show deployment status for an environment."""
    verbose = ctx.obj.get("verbose", False)
    console.print(f"[bold cyan]Status: {env}[/bold cyan]")
    if verbose:
        console.print("  [dim]Fetching from AWS...[/dim]")
    console.print("  [green]✓ api-server  : running[/green]")
    console.print("  [green]✓ worker      : running[/green]")
    console.print("  [yellow]⚠ cache       : 0/3 ready[/yellow]")

# Test with CliRunner (no real shell needed)
runner = CliRunner()
result = runner.invoke(ops_cli, ["--verbose", "status", "--env", "staging"])
print(result.output)

In [ ]:
@ops_cli.command()
@click.argument("service")                    # positional, required
@click.option("--env",   "-e", default="development",
              type=click.Choice(["development", "staging", "production"]))
@click.option("--tag",   "-t", default="latest", help="Docker image tag")
@click.option("--dry-run",     is_flag=True,  help="Print plan, no changes")
def deploy(service, env, tag, dry_run):
    """Deploy a service to the target environment."""
    if dry_run:
        console.print(Panel(
            f"DRY RUN: deploy [bold]{service}:{tag}[/bold] → [italic]{env}[/italic]\n"
            "No changes were made.",
            title="Deploy Plan", border_style="yellow"
        ))
        return
    console.print(f"[green]Deploying {service}:{tag} → {env}[/green]")
    for step in ["Pulling image", "Stopping old containers",
                 "Starting new containers", "Health check"]:
        console.print(f"  [dim]{step}...[/dim]")
    console.print("[bold green]✓ Deployment complete.[/bold green]")

result = runner.invoke(ops_cli,
    ["deploy", "web-api", "--env", "production", "--tag", "v1.2.3", "--dry-run"])
print(result.output)

## Code-Along: Step 3 — Write the CLI as a Standalone Script

In [ ]:
%%writefile ops_cli.py
"""Ops Automation Platform CLI.

Usage:
    python ops_cli.py status --env staging
    python ops_cli.py deploy web-api --tag v1.2.3 --dry-run
    python ops_cli.py scan --target src/
"""
import click
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

@click.group()
@click.option("--verbose", "-v", is_flag=True)
@click.pass_context
def cli(ctx, verbose):
    """Ops Automation Platform CLI."""
    ctx.ensure_object(dict)
    ctx.obj["verbose"] = verbose

@cli.command()
@click.option("--env", "-e", default="development",
              type=click.Choice(["development", "staging", "production"]))
def status(env):
    """Show deployment status."""
    console.print(f"[bold cyan]Status for: {env}[/bold cyan]")
    table = Table()
    table.add_column("Service")
    table.add_column("Status")
    table.add_row("api-server", "[green]running[/green]")
    table.add_row("worker",     "[green]running[/green]")
    table.add_row("cache",      "[yellow]degraded[/yellow]")
    console.print(table)

@cli.command()
@click.argument("service")
@click.option("--env", "-e", default="development",
              type=click.Choice(["development", "staging", "production"]))
@click.option("--tag", "-t", default="latest")
@click.option("--dry-run", is_flag=True)
def deploy(service, env, tag, dry_run):
    """Deploy a service."""
    if dry_run:
        console.print(Panel(f"DRY RUN: {service}:{tag} → {env}", border_style="yellow"))
        return
    console.print(f"[green]Deploying {service}:{tag} → {env}[/green]")

@cli.command()
@click.argument("service")
@click.option("--lines", "-n", default=50, type=int)
def logs(service, lines):
    """Fetch service logs."""
    console.print(f"Fetching last {lines} lines from '{service}'...")

if __name__ == "__main__":
    cli()


In [ ]:
# Verify the script works
import subprocess, sys
r = subprocess.run([sys.executable, "ops_cli.py", "--help"],
                   capture_output=True, text=True)
print(r.stdout)

## Exercise

Add a `scan` sub-command to `ops_cli.py`.

**Your task:**
1. Add `@cli.command()` for `scan`
2. Accept `--target` (default: `.`) and `--port` (default: `80`) options
3. Use `socket.create_connection` to check if the port is open
4. Test it: `runner.invoke(cli, ["scan", "--target", "google.com", "--port", "443"])`

In [ ]:
import socket

# TODO: add @ops_cli.command() for 'scan'
# @ops_cli.command()
# @click.option("--target", default="localhost")
# @click.option("--port", default=80, type=int)
# def scan(target, port):
#     ...

# TODO: test with CliRunner


## Key Takeaways
- Click's `@group` + `@command` pattern mirrors how professional CLIs (kubectl, docker, aws) work
- Always use `click.Choice()` to validate enum-like options — gives free `--help` text
- `Rich` tables and panels require zero formatting logic on your part
- Use `CliRunner` to unit-test CLI commands without a real shell or subprocess

---
**Next:** [Lesson 1.5 — Logging & Structured Logging](lesson-1.5-logging-structured.ipynb)